# Week 3 exercises — Solutions

*Solution notebook. For the best learning experience, do not open this before having attempted and ideally solved the exercise se on your own!*

---

## Part 1: Storing API Keys Securely

### Exercise 1: Creating and using a .env file

a) The `.env` file is created manually with a text editor. Its content should look like this with your key being filled in:

```
# .env (local only, never commit)
GEMINI_API_KEY=xoSKDFLJsdzDPKiodj90DFLSJ         # This fake key is just an example what it could look like
```

b) The `.gitignore` file should contain a line with `.env`.

### Exercise 2: Reading the API key from the .env file

In [1]:
from dotenv import load_dotenv
import os

# Load all variables from .env into the environment
load_dotenv()

api_key = os.environ.get("GEMINI_API_KEY")

if api_key:
    print(f"Key loaded: {api_key[:3]}...")
else:
    print("ERROR: GEMINI_API_KEY not found. Check your .env file.")

Key loaded: AIz...


## Part 2: Loading and Exporting Data

### Exercise 3: Loading CSV data

In [2]:
import pandas as pd

# a) Load the full CSV
df = pd.read_csv("data/cafe_sales.csv")
print("Shape:", df.shape)
print()
df.head()

Shape: (12, 5)



,product,category,units_sold,unit_price,revenue
0,Espresso,Coffee,320,3.5,1120.0
1,Cappuccino,Coffee,280,4.5,1260.0
2,Latte,Coffee,250,4.9,1225.0
3,Green Tea,Tea,180,3.2,576.0
4,Chai Latte,Tea,150,4.7,705.0


In [3]:
# b) Load selected columns with product as index
df_subset = pd.read_csv(
    "data/cafe_sales.csv",
    usecols=["product", "category", "revenue"],
    index_col="product"
)

df_subset

,category,revenue
product,,
Espresso,Coffee,1120.0
Cappuccino,Coffee,1260.0
Latte,Coffee,1225.0
Green Tea,Tea,576.0
Chai Latte,Tea,705.0
Croissant,Pastry,760.0
Blueberry Muffin,Pastry,714.0
Cinnamon Roll,Pastry,585.0
Iced Americano,Coffee,840.0


In [34]:
# c) Export and reload to verify
df_subset.to_csv("data/category_revenue.csv", index=False)

df_check = pd.read_csv("data/category_revenue.csv")
display(df_check)

,category,revenue
0,Coffee,1120.0
1,Coffee,1260.0
2,Coffee,1225.0
3,Tea,576.0
4,Tea,705.0
5,Pastry,760.0
6,Pastry,714.0
7,Pastry,585.0
8,Coffee,840.0
9,Tea,624.0


### Exercise 4: Working with JSON data

In [35]:
import json

# a) Load with the json module
with open("data/projects.json", "r") as f:
    projects = json.load(f)

print("Number of projects:", len(projects))
print()
print("First record:")
print(projects[0])

Number of projects: 3

First record:
{'project_id': 'P001', 'title': 'Website Redesign', 'manager': 'Laura', 'tasks': [{'task': 'Wireframing', 'hours': 20, 'status': 'completed'}, {'task': 'Frontend development', 'hours': 60, 'status': 'in_progress'}, {'task': 'User testing', 'hours': 15, 'status': 'not_started'}]}


In [38]:
# Same thing as above, but done with pandas

projects = pd.read_json("data/projects.json")
projects

,project_id,title,manager,tasks
0,P001,Website Redesign,Laura,"[{'task': 'Wireframing', 'hours': 20, 'status'..."
1,P002,Mobile App Launch,Jukka,"[{'task': 'UI design', 'hours': 35, 'status': ..."
2,P003,Data Pipeline Migration,Sara,"[{'task': 'Schema mapping', 'hours': 15, 'stat..."


In [13]:
# b) Flatten nested tasks using json_normalize
# record_path tells pandas which nested list to expand into rows
# meta keeps parent-level fields alongside each expanded row

df_tasks = pd.json_normalize(
    projects,
    record_path="tasks",
    meta=["project_id", "title"]
)

df_tasks

,task,hours,status,project_id,title
0,Wireframing,20,completed,P001,Website Redesign
1,Frontend development,60,in_progress,P001,Website Redesign
2,User testing,15,not_started,P001,Website Redesign
3,UI design,35,completed,P002,Mobile App Launch
4,Backend API,80,in_progress,P002,Mobile App Launch
5,App store submission,5,not_started,P002,Mobile App Launch
6,Beta testing,25,not_started,P002,Mobile App Launch
7,Schema mapping,15,completed,P003,Data Pipeline Migration
8,ETL scripts,45,completed,P003,Data Pipeline Migration
9,Validation,20,in_progress,P003,Data Pipeline Migration


In [16]:
# c) Export to JSON
df_tasks.to_json("data/tasks_flat.json", orient="records", indent=2)

In [8]:
# c) reload to verify

df_check = pd.read_json("data/tasks_flat.json")
display(df_check)

,task,hours,status,project_id,title
0,Wireframing,20,completed,P001,Website Redesign
1,Frontend development,60,in_progress,P001,Website Redesign
2,User testing,15,not_started,P001,Website Redesign
3,UI design,35,completed,P002,Mobile App Launch
4,Backend API,80,in_progress,P002,Mobile App Launch
5,App store submission,5,not_started,P002,Mobile App Launch
6,Beta testing,25,not_started,P002,Mobile App Launch
7,Schema mapping,15,completed,P003,Data Pipeline Migration
8,ETL scripts,45,completed,P003,Data Pipeline Migration
9,Validation,20,in_progress,P003,Data Pipeline Migration


## Part 3: Databases

### Exercise 5: Connecting and querying a SQLite database

In [40]:
import sqlite3

# a) Connect and list tables
connection = sqlite3.connect("data/retail.db")
db = connection.cursor()

db.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = db.fetchall()

for table in tables:
    print(table[0])

vendor
category
product
region
store
customer
salestransaction
includes


In [42]:
# b) Products costing more than 100€
rows = db.execute(
    """
    SELECT ProductName, ProductPrice 
    FROM Product 
    WHERE ProductPrice > 100
    """
).fetchall()

# print(rows)

for row in rows:
    print(f"{row[0]}: {row[1]}€")

Tiny Tent: 150€
Biggy Tent: 250€
Hi-Tec GPS: 300€
Comfy Harness: 150€
Sunny Charger: 125€
Mega Camera: 275€
Luxo Tent: 500€


In [46]:
# c) Parameterised query — products from vendor "MK"
vendor = "MK"
rows = db.execute(
    "SELECT ProductName FROM Product WHERE VendorID = ?", [vendor]
).fetchall()

for row in rows:
    print(row[0])

Easy Boot
Cosy Sock
Tiny Tent
Biggy Tent
Power Pedals
Comfy Harness
Strongster Carribeaner
Electra Compass


### Exercise 6: Loading database results into pandas

In [12]:
# a) Load the Customer table
df_customers = pd.read_sql_query("SELECT * FROM Customer", connection)

print("Shape:", df_customers.shape)
print()
df_customers.head()

Shape: (10, 3)



,customerid,customername,customerzip
0,1-2-333,Tina,60137
1,2-3-444,Tony,60611
2,3-4-555,Pam,35401
3,4-5-666,Elly,47374
4,5-6-777,Nora,60640


In [13]:
# b) JOIN SalesTransaction and Customer
query = """
    SELECT c.CustomerName, s.TDate, s.StoreID
    FROM SalesTransaction s
    JOIN Customer c ON s.CustomerID = c.CustomerID
"""

df_joined = pd.read_sql_query(query, connection)
df_joined.head()

,customername,tdate,storeid
0,Tina,2020-01-01,S1
1,Tony,2020-01-01,S2
2,Tina,2020-01-02,S3
3,Pam,2020-01-02,S3
4,Tony,2020-01-02,S3


In [14]:
# c) Close the connection
connection.close()

## Part 4: APIs

### Exercise 7: Fetching data from a public API

In [47]:
import requests

# a) Fetch EUR exchange rates
url = "https://open.er-api.com/v6/latest/EUR"
response = requests.get(url)

print("Status code:", response.status_code)

if response.status_code == 200:
    rates = response.json()["rates"]
    print(f"1 EUR = {rates['USD']} USD")
    print(f"1 EUR = {rates['GBP']} GBP")
    print(f"1 EUR = {rates['SEK']} SEK")

Status code: 200
1 EUR = 1.169015 USD
1 EUR = 0.870676 GBP
1 EUR = 10.861204 SEK


In [17]:
# b) Convert rates to a DataFrame
df_rates = pd.DataFrame(
    list(rates.items()),
    columns=["currency", "rate"]
)

df_rates.head(10)

,currency,rate
0,EUR,1.000000
1,AED,4.293193
2,AFN,74.938994
3,ALL,95.855398
4,AMD,439.099465
5,ANG,2.092530
6,AOA,1098.691133
7,ARS,1617.012559
8,AUD,1.654731
9,AWG,2.092530


In [49]:
# c) Open-Meteo 7-day forecast for Turku
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 60.45,
    "longitude": 22.27,
    "daily": "temperature_2m_max,precipitation_sum",
    "timezone": "Europe/Helsinki"
}

response = requests.get(url, params=params)
weather = response.json()

# print(weather)

daily = weather["daily"]

df_weather = pd.DataFrame({
    "date": daily["time"],
    "temp_max": daily["temperature_2m_max"],
    "precipitation": daily["precipitation_sum"]
})

df_weather

,date,temp_max,precipitation
0,2026-04-10,10.5,0.0
1,2026-04-11,11.5,0.0
2,2026-04-12,12.3,0.0
3,2026-04-13,13.6,0.0
4,2026-04-14,15.9,0.0
5,2026-04-15,16.8,0.0
6,2026-04-16,17.6,0.0


### Exercise 8: Using an API key with Gemini

In [19]:
# a) Load the API key

api_key = os.environ.get("GEMINI_API_KEY")

In [42]:
# Installing Gemini if not done before
!pip install google-genai

In [20]:
# b) Send a prompt to Gemini
from google import genai

client = genai.Client(api_key=api_key)
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Explain what an API is in one sentence."
)
print(response.text)

An API (Application Programming Interface) is a set of rules that allows different software applications to communicate and exchange data with each other.


In [22]:
# c) Reusable function
def ask_gemini(prompt):
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt
    )
    return response.text

# Test
print(ask_gemini("What is the capital of Finland?"))

The capital of Finland is **Helsinki**.


With the use of Google's API, you could for example write a script that uses as some component an LLM. As a overly simplistic example, this could for example be used in the sales example to add a new column to a dataset with a brief text description of the product. 

**Note though that you should be careful when using APIs that have billing based on how much you use it. If you do not understand what you are doing, you can easily accumulate expenses.**

## Part 5: Joining and Reshaping Data

### Exercise 9: Merging DataFrames

In [23]:
import pandas as pd

stores = pd.DataFrame({
    "store_id": [1, 2, 3, 4],
    "city": ["Helsinki", "Turku", "Tampere", "Oulu"],
    "region": ["South", "South", "Central", "North"]
})

sales = pd.DataFrame({
    "transaction_id": [1001, 1002, 1003, 1004, 1005],
    "store_id": [1, 2, 2, 3, 5],
    "amount": [250, 120, 310, 180, 90]
})

In [24]:
# a) Inner join
inner = pd.merge(stores, sales, how="inner", on="store_id")
display(inner)
print()
# 4 rows: store_id 1 matches once, store_id 2 matches twice, store_id 3 matches once.
# Store 4 (Oulu) has no sales and store_id 5 has no store, so both are excluded.
print(f"Number of rows: {len(inner)}")

,store_id,city,region,transaction_id,amount
0,1,Helsinki,South,1001,250
1,2,Turku,South,1002,120
2,2,Turku,South,1003,310
3,3,Tampere,Central,1004,180



Number of rows: 4


In [26]:
# b) Left join
left = pd.merge(stores, sales, how="left", on="store_id")
display(left)
print()
# Oulu (store_id 4) has no matching sales, so transaction_id and amount are NaN.
print("Store with no sales: Oulu (store_id 4) — shown with NaN values")

,store_id,city,region,transaction_id,amount
0,1,Helsinki,South,1001.0,250.0
1,2,Turku,South,1002.0,120.0
2,2,Turku,South,1003.0,310.0
3,3,Tampere,Central,1004.0,180.0
4,4,Oulu,North,NaN,NaN



Store with no sales: Oulu (store_id 4) — shown with NaN values


In [27]:
# c) Outer join
outer = pd.merge(stores, sales, how="outer", on="store_id")
display(outer)
print()
# NaN rows:
# - store_id 4 (Oulu): exists in stores but not in sales → transaction_id and amount are NaN
# - store_id 5: exists in sales but not in stores → city and region are NaN
print("Oulu has NaN in sales columns (no transactions)")
print("store_id 5 has NaN in store columns (unknown store)")

,store_id,city,region,transaction_id,amount
0,1,Helsinki,South,1001.0,250.0
1,2,Turku,South,1002.0,120.0
2,2,Turku,South,1003.0,310.0
3,3,Tampere,Central,1004.0,180.0
4,4,Oulu,North,NaN,NaN
5,5,NaN,NaN,1005.0,90.0



Oulu has NaN in sales columns (no transactions)
store_id 5 has NaN in store columns (unknown store)


### Exercise 10: Concatenation and reshaping

In [28]:
# a) Concatenate quarterly DataFrames
q1 = pd.DataFrame({
    "month": ["Jan", "Feb", "Mar"],
    "store_A": [20000, 22000, 21000],
    "store_B": [15000, 16000, 17000]
})

q2 = pd.DataFrame({
    "month": ["Apr", "May", "Jun"],
    "store_A": [23000, 24000, 25000],
    "store_B": [17500, 18000, 19000]
})

combined = pd.concat([q1, q2], ignore_index=True)
combined

,month,store_A,store_B
0,Jan,20000,15000
1,Feb,22000,16000
2,Mar,21000,17000
3,Apr,23000,17500
4,May,24000,18000
5,Jun,25000,19000


In [29]:
# b) Wide to long with melt()
long = combined.melt(
    id_vars="month",
    var_name="store",
    value_name="sales"
)

long

,month,store,sales
0,Jan,store_A,20000
1,Feb,store_A,22000
2,Mar,store_A,21000
3,Apr,store_A,23000
4,May,store_A,24000
5,Jun,store_A,25000
6,Jan,store_B,15000
7,Feb,store_B,16000
8,Mar,store_B,17000
9,Apr,store_B,17500


In [30]:
# c) Long back to wide with pivot()
wide = long.pivot(
    index="month",
    columns="store",
    values="sales"
).rename_axis(columns=None).reset_index()

# Reorder rows to match the original (pivot sorts by index)
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
wide["month"] = pd.Categorical(wide["month"], categories=month_order, ordered=True)
wide = wide.sort_values("month").reset_index(drop=True)

wide

,month,store_A,store_B
0,Jan,20000,15000
1,Feb,22000,16000
2,Mar,21000,17000
3,Apr,23000,17500
4,May,24000,18000
5,Jun,25000,19000
